# DPO 偏好對齊：讓聊天模型學會「哪種回答比較好」

---

## 這份程式要做什麼

上一份 QLoRA（SFT）讓模型**學會對話**。這一份要讓它**學會分辨好壞回答**，使回答更符合人類偏好，這個步驟叫**對齊（alignment）**。

```
① SFT 模型（上一份 QLoRA 訓練出來的，會聊天，但回答品質參差）
        │
        ② 準備「偏好資料」：同一個問題，一個較好的回答 (chosen) 與一個較差的回答 (rejected)
        │
        ③ DPO 訓練：提高「好回答」的機率、降低「壞回答」的機率
        │
        ④ 存下 DPO 的 LoRA 附加層
        ▼
⑤ 合併：SFT 模型 + DPO 附加層 → 最終模型
```

## 先搞懂關鍵詞

### 1. 偏好資料（preference data）

每筆資料是一個三元組：

```
prompt   ：問題（含系統提示與使用者的問題）
chosen   ：比較好的回答
rejected ：比較差的回答
```

### 2. DPO（Direct Preference Optimization，直接偏好最佳化）

以前做對齊要用 **RLHF**：先訓練一個「獎勵模型」去打分，再用強化學習（PPO）調整模型。流程複雜、不穩定、很吃資源。

**DPO** 的想法是：**不需要獎勵模型，也不需要強化學習**，直接用偏好配對做一個簡單的分類式損失。

| | RLHF（PPO） | DPO |
|---|---|---|
| 需要獎勵模型 | 要 | **不要** |
| 訓練流程 | 複雜（多個模型、強化學習） | **接近一般微調** |
| 資源 | 高 | 低 |

### 3. DPO 的損失在做什麼

用文字說明（公式的精確形式我寫在下面，細節以論文為準）：

```
損失 = − log σ( β × [ (模型對好回答的對數機率 − 參考模型對好回答的對數機率)
                    − (模型對壞回答的對數機率 − 參考模型對壞回答的對數機率) ] )
```

直覺：

- **參考模型**（reference model）是訓練開始前的模型，**凍結不動**，當作「原本的樣子」
- 目標是讓模型「相對於原本」**更偏好好回答、更排斥壞回答**
- `β` 控制「可以偏離原本多遠」：`β` 大，模型更保守、離參考模型更近；`β` 小，允許偏離更多

**一個好用的檢查**：訓練剛開始時，模型和參考模型完全一樣，括號裡是 0，`σ(0) = 0.5`，損失 = `−log 0.5 ≈ 0.693`。所以**初始 loss 應該約 0.69**；訓練後 loss 下降，代表模型開始分辨好壞回答。

### 4. 為什麼不需要另外載入參考模型

這份程式用 LoRA 訓練。我印象中 TRL 在這種情況下，會把「**關掉 LoRA 附加層的模型**」當作參考模型，所以不需要多載入一份。這點我沒有針對你的版本確認。

### 5. 為什麼通常「先 SFT、再 DPO」

DPO 假設模型本來就會回答問題，只是要微調「回答的偏好」。如果基礎模型連對話格式都不會，直接做 DPO 效果通常不好。所以這份程式的起點是上一份訓練好的 SFT 模型。

## 兩層 LoRA 的整體流程

```
基礎模型 TinyLlama
   + SFT 的 LoRA（Qlora_Finetune_model）      ← 上一份產生的
   = 合併 → SFT 模型
        + DPO 的 LoRA（Dpo_Finetune_model）    ← 這一份產生的
        = 合併 → 最終模型（會聊天，且偏好更好的回答）
```

## 從報錯學到的事

你前面遇到並修掉的兩個問題：

| 問題 | 原因 | 修法 |
|---|---|---|
| `ValueError: You passed a PeftModel instance together with a peft_config` | 自己用 `get_peft_model` 包了 LoRA，又傳 `peft_config` 叫訓練器再包一次 | 刪掉 `get_peft_model`，傳**普通模型**加上 `peft_config` |
| 合併後的 `merged_model` 沒被使用 | 先前寫成 `prepare_model_for_kbit_training(model)`，用了合併前的 `PeftModel` | 改成 `prepare_model_for_kbit_training(merged_model)` |

**規則**：`peft_config` 的意思是「請幫我包 LoRA」。模型已經包過了，就不能再傳；要傳 `peft_config`，就給它一個還沒包過的普通模型。

## 硬體提醒

- 需要 **NVIDIA GPU**（4 位元量化與 bf16）。
- 需要先有上一份 SFT 存下來的資料夾 `Qlora_Finetune_model`（見「觀察到的地方」第 1 點，資料夾名稱要對得上）。

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入偏好資料集 |
| `AutoPeftModelForCausalLM` | 載入「基礎模型 + LoRA 附加層」 |
| `BitsAndBytesConfig` | 4 位元量化設定 |
| `LoraConfig` | 設定新的 LoRA（給 DPO 訓練用） |
| `prepare_model_for_kbit_training` | 為「量化模型的訓練」做準備 |
| `DPOConfig`、`DPOTrainer`（來自 `trl`） | DPO 專用的設定與訓練器 |
| `PeftModel` | 把一個 LoRA 附加層掛到指定的模型上 |
| `pipeline` | 一行程式做文字生成 |

`get_peft_model` 有匯入但現在**沒有用到**（因為由 `DPOTrainer` 自動套用 LoRA）。

In [ ]:
from datasets import load_dataset
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer
from peft import PeftModel
from transformers import pipeline

## 步驟 1：載入並過濾偏好資料集

```python
load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
```

### 這個資料集是什麼

它是常用偏好資料集 `Intel/orca_dpo_pairs` 的「**重新標註版**」。原版假設「GPT-4 的回答一定比 GPT-3.5 好」，但這未必成立；所以 Argilla 用評審模型重新評分，產生更可靠的偏好標籤。

重點欄位（資料集頁面上的說明，加上我的印象）：

| 欄位 | 說明 |
|---|---|
| `system` | 系統提示 |
| `input` | 使用者的問題 |
| `chosen` | 較好的回答 |
| `rejected` | 較差的回答 |
| `status` | 評審後的結果：`unchanged`（維持原本的好壞）、`swapped`（評審認為原本標反了，已對調）、`tie`（兩者不分高下） |
| `chosen_score` | 評審給 `chosen` 的分數 |
| `in_gsm8k_train` | 這個問題是否和 GSM8K（數學題資料集）的訓練題重複 |

另外還有評分過程的欄位（例如 `rating`、`generations` 等），這份程式不需要。

### 過濾條件

```python
dpo_dataset.filter(
    lambda example:
    example["status"] != "tie" and
    example["chosen_score"] >= 8 and
    not example["in_gsm8k_train"]
)
```

| 條件 | 意思 | 為什麼 |
|---|---|---|
| `status != "tie"` | 排除「不分高下」的配對 | 沒有明確好壞，無法提供學習訊號 |
| `chosen_score >= 8` | 只留好回答的評分在 8 分以上 | 確保「好回答」真的是高品質 |
| `not in_gsm8k_train` | 排除和 GSM8K 訓練題重複的資料 | 我推測是為了避免日後在 GSM8K 上評測時產生資料污染 |

### 這個 lambda 怎麼讀

`lambda example: 條件` 是「丟一筆資料進去，回傳 True 就保留、False 就丟掉」。三個條件用 `and` 連起來，**全部成立才保留**。

過濾後剩下的筆數我印象中約六、七千筆，請用 `len(dpo_dataset)` 確認。

In [ ]:
dpo_dataset = load_dataset(
    "argilla/distilabel-intel-orca-dpo-pairs",
    split="train",  
)

dpo_dataset = dpo_dataset.filter(
    lambda example:
    example["status"] != "tie" and
    example["chosen_score"] >= 8 and
    not example["in_gsm8k_train"]
)

## 步驟 2：整理成 DPOTrainer 要的格式

`DPOTrainer` 需要資料集有三個欄位：`prompt`、`chosen`、`rejected`。`format_prompt` 就是把原始欄位組成這三個字串。

```python
system = "<|system|> \n" + example["system"] + "</s> \n"
prompt = "<|user|> \n" + example["input"] + "</s> \n <|assistant|> \n"
chosen = example["chosen"] + "</s> \n"
rejected = example["rejected"] + "</s> \n"
```

組出來的樣子（示意）：

```
prompt   = "<|system|> \n{系統提示}</s> \n<|user|> \n{使用者的問題}</s> \n <|assistant|> \n"
chosen   = "{好回答}</s> \n"
rejected = "{壞回答}</s> \n"
```

| 欄位 | 內容 |
|---|---|
| `prompt` | 系統提示 + 使用者的問題 + `<|assistant|>`（把「該輪到助理回答了」標出來） |
| `chosen` | **只有助理的回答**，結尾加上結束符號 |
| `rejected` | 同上，是較差的回答 |

`<|system|>`、`<|user|>`、`<|assistant|>`、`</s>` 是 TinyLlama 聊天模型使用的角色標記，和上一份 SFT 是同一種對話格式。

### `remove_columns=dpo_dataset.column_names`

和上一份一樣：處理完之後把**原本所有欄位都移除**，只留下 `prompt`、`chosen`、`rejected` 三欄，避免多餘欄位干擾訓練器的格式判斷。`column_names` 是在 `map` 被呼叫前就先取值的。

### 這份資料的格式和 SFT 不同

| | SFT（上一份） | DPO（這一份） |
|---|---|---|
| 每筆資料 | 一整段對話文字 `text` | 三個欄位：`prompt`、`chosen`、`rejected` |
| 學習目標 | 模仿「好回答」 | 比較「好回答 vs 壞回答」 |

> 檢查資料的好方法：把被註解掉的 `# print(dpo_dataset[0])` 打開，看看三個欄位長什麼樣。

In [ ]:
def format_prompt(example):
    system = "<|system|> \n" + example["system"] + "</s> \n"
    prompt = "<|user|> \n" + example["input"] + "</s> \n <|assistant|> \n" 
    chosen = example["chosen"] + "</s> \n"
    rejected = example["rejected"] + "</s> \n"
    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }


dpo_dataset = dpo_dataset.map(
    format_prompt,
    remove_columns=dpo_dataset.column_names,
)

# print(dpo_dataset[0])

## 步驟 3：載入上一份的 SFT 模型並合併

```python
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.bfloat16
)
```

這是和上一份相同的 4 位元量化設定（NF4、雙重量化、運算時用 bf16）。

```python
model = AutoPeftModelForCausalLM.from_pretrained(
    "Qlora_Finetune_model", ..., quantization_config = bnb_config,
)
merged_model = model.merge_and_unload()
```

| 步驟 | 發生什麼事 |
|---|---|
| `AutoPeftModelForCausalLM.from_pretrained("Qlora_Finetune_model")` | 讀取 SFT 的 LoRA 資料夾，自動載入它掛接的基礎模型（這次**以 4 位元載入**），再接上 SFT 附加層 |
| `merge_and_unload()` | 把 SFT 附加層合併進權重，得到一個**普通的 4 位元模型**，裡面已經包含 SFT 學到的對話能力 |

為什麼先合併？因為接下來要在上面加**新的** LoRA（DPO 用的）。已經有附加層的模型不能再叫訓練器多加一層，所以要先合併成普通模型。

> `merge_and_unload()` 把 LoRA 合進**已經壓成 4 位元**的權重，我印象中 PEFT 會警告這可能產生捨入誤差。這是這種做法的已知取捨。

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoPeftModelForCausalLM.from_pretrained(
    "Qlora_Finetune_model",
    device_map="auto",
    low_cpu_mem_usage=True,
    quantization_config = bnb_config,
)
merged_model = model.merge_and_unload()

## 步驟 4：載入分詞器

```python
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
```

| 設定 | 說明 |
|---|---|
| `model_name` | 基礎模型的名稱，用它的分詞器（SFT 時也是同一個詞彙表） |
| `trust_remote_code=True` | 允許執行來源的自訂程式碼。Llama 架構其實不需要，**只應該對信任的來源開啟** |
| `pad_token = eos_token` | 這個分詞器沒有補齊符號，借用結束符號 |
| `padding_side = "left"` | 補齊補在左邊。上一份 SFT 訓練用的是靠右；這份沿用靠左，我沒有確認對 DPO 訓練的影響 |

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code = True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

## 步驟 5：設定新的 LoRA，並為量化訓練做準備

### 新的 LoRA 設定

```python
peft_config = LoraConfig(r=64, lora_alpha=32, target_modules=[...7 種線性層...], ...)
```

參數和上一份 SFT 相同（`r = 64`、`lora_alpha = 32`、七種線性層都加）。這是要給 **DPO 訓練**用的**新附加層**，和 SFT 的附加層是分開的兩組。

### `prepare_model_for_kbit_training`

```python
model_for_dpo = prepare_model_for_kbit_training(merged_model)
```

這是替「量化模型的訓練」做前置準備。我印象中它做的事包含：把不被量化的部分（例如正規化層）轉成較高精度以求數值穩定，以及讓梯度檢查點可以正常運作。

### 為什麼沒有 `get_peft_model`

你之前的版本在這裡自己呼叫了 `get_peft_model`，再加上 `peft_config` 傳給 `DPOTrainer`，就會得到那個 `ValueError`。現在的做法：

| 做法 | 傳給 DPOTrainer 的模型 | `peft_config` |
|---|---|---|
| **現在這份（做法 A）** | **普通模型 `model_for_dpo`** | **傳** |
| 做法 B（自己加 LoRA） | `get_peft_model(...)` 的結果 | **不傳** |

兩者擇一，不能同時做。

In [ ]:
peft_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=["k_proj", "gate_proj", "up_proj", "q_proj", "v_proj", "o_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model_for_dpo = prepare_model_for_kbit_training(merged_model)

## 步驟 6：DPO 訓練設定（DPOConfig）

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `"Dpo_results"` | 訓練輸出資料夾 |
| `per_device_train_batch_size` | 2 | 每次放 2 筆進 GPU |
| `gradient_accumulation_steps` | 4 | 累積 4 次才更新，**等效 batch = 8** |
| `optim` | `"paged_adamw_32bit"` | 分頁版 AdamW，記憶體吃緊時暫時換到 CPU |
| `learning_rate` | `1e-5` | **很小**，比上一份 SFT 的 `2e-4` 小 20 倍。偏好微調很敏感，學習率太大容易把模型訓壞（我依經驗與印象寫的） |
| `lr_scheduler_type` | `"cosine"` | 學習率依餘弦曲線下降 |
| `max_steps` | 200 | **只訓練 200 次更新**，會**覆蓋** `num_train_epochs` |
| `logging_steps` | 10 | 每 10 步印一次記錄 |
| `bf16` / `fp16` | True / False | bf16 混合精度（要和量化的 `compute_dtype` 一致） |
| `gradient_checkpointing` | True | 用重算換記憶體 |
| `max_length` | 512 | 序列最長 512 個 token，超過截斷 |
| `beta` | 0.1 | DPO 的 β，控制模型可以偏離參考模型多遠 |

### 訓練量速算

```
200 次更新 × 等效 batch 8 = 1600 筆偏好配對
```

過濾後約有六、七千筆資料，所以這次只看了其中約四分之一。這是示範規模。

### 為什麼用 `max_steps` 而不是 epoch

用步數來控制比較直接：想快速試跑，就用小的步數，不必管資料集有幾筆。

### `beta` 怎麼影響結果

| `beta` | 效果 |
|---|---|
| 小（例如 0.05） | 模型被允許大幅偏離原本的樣子，偏好學得快，但可能不穩定 |
| 大（例如 0.5） | 模型被限制得更貼近原本的樣子，學得保守 |
| `0.1` | 常見的預設值 |

這個敘述是我依 DPO 論文的印象整理的。

> 你的版本把原本被註解掉的 `warmup_ratio` 和 `max_prompt_length` 都拿掉了，沒有影響（沒設定時用預設值）。

In [ ]:
training_arguments = DPOConfig(
    output_dir = "Dpo_results",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    optim = "paged_adamw_32bit",
    learning_rate = 1e-5,
    lr_scheduler_type = "cosine",
    max_steps = 200,
    logging_steps = 10,
    bf16 = True,
    fp16 = False,
    gradient_checkpointing = True,
    max_length = 512,
    beta = 0.1,
)

## 步驟 7：建立 DPOTrainer

```python
dpo_trainer = DPOTrainer(
    model_for_dpo,
    args = training_arguments,
    train_dataset = dpo_dataset,
    processing_class = tokenizer,
    peft_config = peft_config,
)
```

| 參數 | 說明 |
|---|---|
| 第一個參數 `model_for_dpo` | 普通的 4 位元模型（已含 SFT 能力），**還沒有 LoRA** |
| `args` | 剛剛的 `DPOConfig` |
| `train_dataset` | 有 `prompt`、`chosen`、`rejected` 三欄的資料集 |
| `processing_class` | 分詞器（新版名稱；舊版叫 `tokenizer`） |
| `peft_config` | 請訓練器幫忙包上新的 LoRA |

沒有傳 `ref_model`（參考模型）：使用 LoRA 時，我印象中 TRL 會用「關掉附加層的模型」當參考，不需要另外載入。

In [ ]:
dpo_trainer = DPOTrainer(
    model_for_dpo,
    args = training_arguments,
    train_dataset = dpo_dataset,
    processing_class = tokenizer,
    peft_config = peft_config,
)

## 步驟 8：訓練並儲存 DPO 附加層

```python
dpo_trainer.train()
dpo_trainer.save_model("Dpo_Finetune_model")
```

### 訓練時的記錄怎麼看

每 10 步會印出一次記錄。DPO 的記錄除了 `loss`，我印象中還有幾個和「獎勵」有關的指標（名稱依版本而定）：

| 指標（印象） | 意思 | 好的趨勢 |
|---|---|---|
| `loss` | DPO 損失，初始約 **0.69** | 逐漸**下降** |
| `rewards/chosen` | 模型對好回答的「隱含獎勵」 | 上升 |
| `rewards/rejected` | 模型對壞回答的「隱含獎勵」 | 下降 |
| `rewards/margins` | 好回答與壞回答的獎勵差 | **上升（變大）** |
| `rewards/accuracies` | 模型把好回答排得比壞回答高的比例 | **上升，超過 0.5** |

初始時 `accuracies` 約 0.5（等於隨機猜）、`margins` 約 0，訓練後這兩個指標變好，就代表模型在學習分辨好壞。

### 存的是什麼

只存 **DPO 的 LoRA 附加層**，不是完整模型。它必須搭配「SFT 模型」才有意義，這也是下一步要把兩者合併的原因。

In [ ]:
dpo_trainer.train()
dpo_trainer.save_model("Dpo_Finetune_model")

## 步驟 9：合併 SFT 與 DPO，得到最終模型

```python
# ① 重新載入 SFT 模型（這次不量化），合併
model = AutoPeftModelForCausalLM.from_pretrained("Qlora_Finetune_model", ...)
sft_model = model.merge_and_unload()

# ② 把 DPO 附加層掛上去，再合併
dpo_model = PeftModel.from_pretrained(sft_model, "Dpo_Finetune_model", device_map = "auto")
dpo_model = dpo_model.merge_and_unload()
```

### 為什麼要重新載入 SFT 模型

| | 步驟 3 的 `merged_model` | 這裡的 `sft_model` |
|---|---|---|
| 精度 | 4 位元 | **一般精度（沒指定量化）** |
| 用途 | 給 DPO **訓練**用（省記憶體） | 給最終**合併與使用**（較精確） |

最終模型想要精確一點，所以用**沒有量化**的版本重新合併一次。

### 合併的順序

```
基礎模型 + SFT 附加層 ──merge──▶ sft_model
sft_model + DPO 附加層 ──merge──▶ dpo_model   ← 同時含有 SFT 與 DPO 的學習成果
```

`PeftModel.from_pretrained(模型, 資料夾)` 的意思是：「把這個資料夾裡的 LoRA 附加層，掛到這個模型上」。

### 一個要小心的地方：`sft_model` 會被連帶改掉

`PeftModel.from_pretrained(sft_model, ...)` 和後續的 `merge_and_unload()`，是**直接在 `sft_model` 的權重上做修改**（這是我依 PEFT 的運作方式推論的，沒有實測）。所以做完之後，`sft_model` 裡**已經包含了 DPO 的成果**，不再是純 SFT 版本。

**如果想比較「SFT 前後 DPO」的差別，必須在掛上 DPO 附加層之前，先用 `sft_model` 生成並記下結果。**

### 訓練與最終合併的精度落差

DPO 附加層是在「**4 位元合併版**」的 SFT 模型上訓練的，卻疊到「**一般精度合併版**」上使用，兩者之間有細微的落差。這是這種流程常見的取捨，影響多大我沒有實測。

### 顯存提醒（小 GPU）

這一格會再載入一份完整的模型，而訓練用的模型（`model_for_dpo`、`dpo_trainer`）可能還留在記憶體裡。8 GB 的 GPU 可能吃緊。遇到 `CUDA out of memory` 時，可以先釋放：

```python
import gc
del dpo_trainer, model_for_dpo, merged_model, model
gc.collect()
torch.cuda.empty_cache()
```

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "Qlora_Finetune_model",
    device_map="auto",
    low_cpu_mem_usage=True,
)
sft_model = model.merge_and_unload()

dpo_model = PeftModel.from_pretrained(
    sft_model,
    "Dpo_Finetune_model",
    device_map = "auto",
)
dpo_model = dpo_model.merge_and_unload()

## 步驟 10：用最終模型生成回答

```python
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
```

使用者提問，後面接上 `<|assistant|>`，讓模型接著寫助理的回答。

```python
pipe = pipeline(task="text-generation", model=dpo_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])
```

| 部分 | 說明 |
|---|---|
| 輸出內容 | **包含 prompt 加上模型生成的續寫** |
| 長度 | 沒有指定生成長度，如果被截得很短，加上 `max_new_tokens=256` |
| 隨機性 | 預設通常是貪婪解碼，同樣輸入每次輸出相同；想多樣化加 `do_sample=True`、`temperature=0.7` |

### 注意格式的一致性

這個 prompt 的格式是 `<|user|>\n...</s>\n<|assistant|>\n`（沒有系統提示，換行前後**沒有多餘空格**）。而訓練資料（步驟 2）是 `"<|user|> \n" ... "</s> \n <|assistant|> \n"`（換行前**有空格**）。空白不同，分詞後的 token 就不同。影響多大我沒有確認，要最保險的話，推論時用和訓練相同的格式。

### 怎麼判斷 DPO 有沒有效

DPO 的改變通常**很細微**（回答的語氣、完整度、是否更貼題）。最好的方法是用**同一個 prompt**，比較「SFT 模型」和「DPO 模型」的輸出，而且要多換幾個問題。我沒有實測，請以你跑出來的結果為準。

In [ ]:
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

pipe = pipeline(task="text-generation", model=dpo_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

---

# 重點整理

## 一、整份程式一句話

> 以上一份 QLoRA 訓練好的 SFT 模型為起點，用過濾後的偏好資料（問題、好回答、壞回答）做 200 步的 DPO 訓練，只更新新加的 LoRA；最後把 SFT 與 DPO 兩層附加層依序合併，得到更符合偏好的聊天模型。

## 二、必須搞懂的 11 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 對齊（alignment） | 讓模型的回答更符合人類偏好 |
| 2 | 偏好配對 | 同一個問題的「好回答（chosen）」與「壞回答（rejected）」 |
| 3 | RLHF | 傳統做法：訓練獎勵模型 + 強化學習，複雜 |
| 4 | DPO | 不需要獎勵模型，直接用偏好配對訓練，接近一般微調 |
| 5 | 參考模型 | 訓練前的模型，凍結不動，作為「原本的樣子」 |
| 6 | `beta` | 控制模型能偏離參考模型多遠 |
| 7 | 初始 loss ≈ 0.69 | 模型與參考模型相同時，損失是 `−log 0.5` |
| 8 | 先 SFT 再 DPO | DPO 假設模型本來就會回答問題 |
| 9 | 附加層疊加 | SFT 的 LoRA 先合併，再訓練並合併 DPO 的 LoRA |
| 10 | `peft_config` 的語意 | 「請幫我包 LoRA」，只能給還沒包過的普通模型 |
| 11 | `merge_and_unload` | 把 LoRA 加進權重，變回普通模型，且是原地修改 |

## 三、程式流程速記

```
1. load_dataset + filter           → 去掉 tie、chosen_score < 8、GSM8K 重複題
2. format_prompt + map             → prompt / chosen / rejected 三欄，其餘欄位移除
3. 載入 SFT 模型（4-bit）+ merge   → 得到含 SFT 能力的普通 4 位元模型
4. tokenizer                       → pad = eos、靠左
5. LoraConfig + prepare_model_for_kbit_training → 準備訓練
6. DPOConfig                       → lr 1e-5、200 步、beta 0.1、bf16
7. DPOTrainer + train + save       → 只訓練新的 LoRA，存 DPO 附加層
8. 重載 SFT（不量化）+ merge → 掛 DPO 附加層 + merge → 最終模型
9. pipeline 生成                   → 用對話格式測試
```

## 四、最容易搞混的地方

1. **為什麼 DPO 不需要獎勵模型？**
   它把「獎勵」隱含在模型與參考模型的機率比值裡，直接用偏好配對做損失。

2. **`beta` 越大越好嗎？**
   不是。`beta` 大，模型更保守，離原本更近；`beta` 小，學得更激進但可能不穩定。

3. **為什麼學習率這麼小（1e-5）？**
   偏好微調很敏感，太大的學習率容易把模型訓壞，比 SFT 的 2e-4 小很多。

4. **為什麼不能自己 `get_peft_model` 又傳 `peft_config`？**
   `peft_config` 會叫訓練器再包一次 LoRA，疊兩層會出錯。

5. **為什麼訓練時載入 4 位元，最終合併時又重新載入一般精度？**
   訓練求省記憶體，最終合併求精確。

6. **步驟 9 之後 `sft_model` 還是純 SFT 嗎？**
   不是。掛上 DPO 附加層並合併後，`sft_model` 的權重已被連帶修改。

7. **資料格式和 SFT 有什麼不同？**
   SFT 是單一欄位 `text`；DPO 是 `prompt`、`chosen`、`rejected` 三欄。

8. **為什麼要先過濾資料？**
   去掉不分高下的配對、確保好回答品質、避免和數學評測資料重複。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `ValueError: You passed a PeftModel instance together with a peft_config` | 模型已被包成 `PeftModel`，又傳了 `peft_config` | 傳普通模型 + `peft_config`，或傳 `PeftModel` 且不傳 `peft_config` |
| 找不到 `Qlora_Finetune_model` | SFT 存檔的資料夾名稱不同（例如先前存成 `LoRA_Fineune_model`） | 確認資料夾名稱，或把程式裡的路徑改成實際名稱 |
| `CUDA out of memory` | 同時載入多個模型、序列太長 | 降低 `max_length`、在載入最終模型前先釋放訓練用的物件 |
| `KeyError`（欄位不存在） | 資料集欄位名稱和預期不同 | 印出 `dpo_dataset.column_names` 檢查 |
| 訓練 loss 不下降 | 學習率不合適、資料格式有誤、`beta` 設定不當 | 檢查 `print(dpo_dataset[0])`、調整學習率與 `beta` |
| 生成的回答和 SFT 幾乎一樣 | 只訓練 200 步，改變很細微 | 多換幾個問題比較，或增加步數 |
| bf16 相關錯誤 | GPU 不支援 bf16 | 改用 `fp16`，並把 `compute_dtype` 改回 `torch.float16` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **SFT 資料夾名稱要對得上**
   這份程式載入 `"Qlora_Finetune_model"`，但先前的 SFT 程式存的是 `"LoRA_Fineune_model"`。如果你有改名沒問題，否則會找不到資料夾。

2. **提示格式的空白不一致**
   訓練資料的換行前有空格（`"</s> \n <|assistant|> \n"`），推論用的 prompt 沒有空格，和 SFT 的對話模板也略有差異。

3. **合併到 4 位元權重有捨入誤差**
   而且 DPO 是在 4 位元合併版上訓練、卻疊到一般精度合併版上使用。

4. **`get_peft_model` 匯入了但沒用到**
   不影響執行。

5. **`sft_model` 會被連帶修改**
   想比較 SFT 與 DPO 的輸出，必須在掛上 DPO 附加層之前先生成。

6. **訓練量很小**
   200 步、約 1600 筆配對，是示範規模，DPO 的效果可能很細微。

7. **沒有驗證資料**
   只能靠訓練記錄中的 rewards 指標和生成結果主觀判斷。

8. **`padding_side = "left"`**
   和上一份 SFT 訓練的靠右不同，對 DPO 的影響我沒有確認。

9. **`trust_remote_code=True` 不一定需要**
   只應對可信任的來源開啟。

10. **偏好標籤來自 AI 評審**
    `chosen_score` 是評審模型給的分數，不是人類標註，評審本身也可能有偏誤。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 檢查整理後的資料
print(dpo_dataset.column_names)      # 預期 ['prompt', 'chosen', 'rejected']
print(len(dpo_dataset))
print(dpo_dataset[0]["prompt"])
print(dpo_dataset[0]["chosen"][:200])
print(dpo_dataset[0]["rejected"][:200])

# 2. 在合併 DPO 之前，先記下 SFT 模型的輸出（之後才能公平比較）
pipe_sft = pipeline(task="text-generation", model=sft_model, tokenizer=tokenizer)
print(pipe_sft(prompt, max_new_tokens=150)[0]["generated_text"])
#    …然後才執行 PeftModel.from_pretrained 與 merge_and_unload

# 3. 調整 beta（0.05、0.1、0.5），比較訓練記錄與生成結果

# 4. 把 max_steps 從 200 增加到 500，觀察 rewards/accuracies 是否更高

# 5. 觀察訓練記錄裡 rewards/accuracies 與 rewards/margins 的趨勢

# 6. 存下最終完整模型，之後不依賴 peft 也能直接使用
dpo_model.save_pretrained("sft_dpo_merged")
tokenizer.save_pretrained("sft_dpo_merged")
```

## 八、SFT 與 DPO 的對照

| | SFT（上一份 QLoRA） | **DPO（這份）** |
|---|---|---|
| 目標 | 學會對話格式與回答 | **偏好更好的回答** |
| 資料 | 對話文字 `text` | **`prompt` + `chosen` + `rejected`** |
| 學習方式 | 模仿範例 | **比較好壞** |
| 是否需要參考模型 | 不需要 | **需要（LoRA 時用關掉附加層的模型）** |
| 學習率 | `2e-4` | **`1e-5`** |
| 訓練量 | 3000 筆、1 epoch（約 375 次更新） | **200 次更新（約 1600 筆配對）** |
| 起點 | 基礎模型 | **SFT 模型** |
| 產出 | SFT 的 LoRA | **DPO 的 LoRA（疊在 SFT 之上）** |

## 九、完整的訓練路線圖（你這一系列學到的）

```
預訓練（別人做好）→ 基礎模型
        │
        ├─ 全參數微調（影評分類、NER、MLM）  ← 前面學的
        │
        ▼ SFT（QLoRA）：學會對話
        ▼ DPO：學會偏好更好的回答             ← 這一份
        ▼ 對齊後的聊天模型
```

## 十、一句話記住

> **DPO = 給模型一堆「好回答 vs 壞回答」的配對，讓它在不偏離原本太遠的前提下，更偏好好回答。** 它省去了獎勵模型與強化學習，用接近一般微調的流程完成對齊。